In [ ]:
# PASO 1: Carga y visualización del dataset

import pandas as pd
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "datos").exists() and (candidate / "resultados").exists():
            return candidate
    raise FileNotFoundError("No se pudo localizar la raíz del repositorio.")

REPO_ROOT = find_repo_root(Path.cwd())
RUTA_SCORING_MACRO = REPO_ROOT / "resultados" / "05_scoring" / "scoring_macro.csv"
RUTA_RESULTADOS_ESCENARIOS = REPO_ROOT / "resultados" / "06_escenarios"
RUTA_RESULTADOS_ESCENARIOS.mkdir(parents=True, exist_ok=True)

if RUTA_SCORING_MACRO.exists():
    df_esc = pd.read_csv(RUTA_SCORING_MACRO)
    print(f"✅ Archivo macro cargado correctamente: {df_esc.shape[0]} zonas detectadas.")
    print(f"📂 Ruta utilizada: {RUTA_SCORING_MACRO}")
else:
    raise FileNotFoundError(f"❌ ERROR: No se encontró el archivo en {RUTA_SCORING_MACRO}")

display(df_esc.head())


In [ ]:
# PASO 2: ESCENARIO POTENCIAL DE DEMANDA (35% Demanda, 25% POIs)

# 2.1 Cálculo del escenario
df_esc["ESC_POTENCIAL_DEMANDA"] = (
    df_esc["DEMANDA"] * 0.35 +
    df_esc["PTOS_INTERES"] * 0.25 +
    df_esc["MOVILIDAD"] * 0.15 +
    df_esc["SEGURIDAD"] * 0.10 +
    df_esc["COSTE"] * 0.10 +
    df_esc["COMPETENCIA"] * 0.05
).round(2)

# 2.2 Cálculo de ranking
df_esc["RANK_DEMANDA"] = df_esc["ESC_POTENCIAL_DEMANDA"].rank(ascending=False, method="min").astype(int)

# 2.3 Visualización completa
pd.set_option("display.max_rows", None)
print(" RANKING: ESCENARIO POTENCIAL DE DEMANDA")
display(
    df_esc[["RANK_DEMANDA", "ID_ZONA", "NOMBRE_ZONA", "CLUSTER", "ESC_POTENCIAL_DEMANDA"]]
    .sort_values(by="ESC_POTENCIAL_DEMANDA", ascending=False)
)
pd.reset_option("display.max_rows")

# 2.4 Exportación del escenario
ruta_escenario_demanda = RUTA_RESULTADOS_ESCENARIOS / "escenario_potencial_demanda.csv"
df_esc.to_csv(ruta_escenario_demanda, index=False)
print(f"✅ Escenario 1 exportado exitosamente en: {ruta_escenario_demanda}")


In [ ]:
# PASO 3: ESCENARIO EFICIENCIA Y FLUJO (35% Movilidad, 25% POIs)

# 3.1 Cálculo del escenario
df_esc["ESC_EFICIENCIA_FLUJO"] = (
    df_esc["MOVILIDAD"] * 0.35 +
    df_esc["PTOS_INTERES"] * 0.25 +
    df_esc["DEMANDA"] * 0.15 +
    df_esc["SEGURIDAD"] * 0.10 +
    df_esc["COSTE"] * 0.10 +
    df_esc["COMPETENCIA"] * 0.05
).round(2)

# 3.2 Cálculo de ranking
df_esc["RANK_FLUJO"] = df_esc["ESC_EFICIENCIA_FLUJO"].rank(ascending=False, method="min").astype(int)

# 3.3 Visualización completa
pd.set_option("display.max_rows", None)
print(" RANKING: ESCENARIO EFICIENCIA Y FLUJO")
display(
    df_esc[["RANK_FLUJO", "ID_ZONA", "NOMBRE_ZONA", "CLUSTER", "ESC_EFICIENCIA_FLUJO"]]
    .sort_values(by="ESC_EFICIENCIA_FLUJO", ascending=False)
)
pd.reset_option("display.max_rows")

# 3.4 Exportación del escenario
ruta_escenario_flujo = RUTA_RESULTADOS_ESCENARIOS / "escenario_eficiencia_flujo.csv"
df_esc.to_csv(ruta_escenario_flujo, index=False)
print(f"✅ Escenario 2 exportado exitosamente en: {ruta_escenario_flujo}")


In [ ]:
# PASO 4: ESCENARIO VIABILIDAD Y RIESGO (20% Seguridad, 25% Coste, 15% Competencia)

# 4.1 Cálculo del escenario
df_esc["ESC_VIABILIDAD_RIESGO"] = (
    df_esc["SEGURIDAD"] * 0.20 +
    df_esc["COSTE"] * 0.25 +
    df_esc["COMPETENCIA"] * 0.15 +
    df_esc["DEMANDA"] * 0.15 +
    df_esc["MOVILIDAD"] * 0.10 +
    df_esc["PTOS_INTERES"] * 0.15
).round(2)

# 4.2 Cálculo de ranking
df_esc["RANK_VIABILIDAD"] = df_esc["ESC_VIABILIDAD_RIESGO"].rank(ascending=False, method="min").astype(int)

# 4.3 Visualización completa
pd.set_option("display.max_rows", None)
print(" RANKING: ESCENARIO VIABILIDAD Y RIESGO")
display(
    df_esc[["RANK_VIABILIDAD", "ID_ZONA", "NOMBRE_ZONA", "CLUSTER", "ESC_VIABILIDAD_RIESGO"]]
    .sort_values(by="ESC_VIABILIDAD_RIESGO", ascending=False)
)
pd.reset_option("display.max_rows")

# 4.4 Exportación del escenario
ruta_escenario_viabilidad = RUTA_RESULTADOS_ESCENARIOS / "escenario_viabilidad_riesgo.csv"
df_esc.to_csv(ruta_escenario_viabilidad, index=False)
print(f"✅ Escenario 3 exportado exitosamente en: {ruta_escenario_viabilidad}")


In [ ]:
# PASO 5: CONSISTENCIA INTERNA Y RANKING GLOBAL

# 5.1 Score medio de los tres escenarios
df_esc["SCORE_MEDIO"] = (
    df_esc["ESC_POTENCIAL_DEMANDA"] +
    df_esc["ESC_EFICIENCIA_FLUJO"] +
    df_esc["ESC_VIABILIDAD_RIESGO"]
) / 3
df_esc["SCORE_MEDIO"] = df_esc["SCORE_MEDIO"].round(2)

# 5.2 Variabilidad entre escenarios (desviación estándar de los ranks)
df_esc["VARIABILIDAD_ENTRE_ESCENARIOS"] = (
    df_esc[["RANK_DEMANDA", "RANK_FLUJO", "RANK_VIABILIDAD"]]
    .std(axis=1, ddof=0)
    .round(2)
)

# 5.3 Clasificación interpretativa de la consistencia
def clasificar_consistencia(x):
    if x <= 3:
        return "Alta consistencia"
    elif x <= 6:
        return "Consistencia media"
    else:
        return "Baja consistencia"

df_esc["CONSISTENCIA_ESCENARIOS"] = df_esc["VARIABILIDAD_ENTRE_ESCENARIOS"].apply(clasificar_consistencia)

# 5.4 Ranking global basado en score medio
df_esc["RANKING_GLOBAL"] = (
    df_esc["SCORE_MEDIO"]
    .rank(ascending=False, method="min")
    .astype(int)
)

# 5.5 Visualización completa
pd.set_option("display.max_rows", None)
print(" RANKING GLOBAL Y CONSISTENCIA INTERNA")
display(
    df_esc[[
        "RANKING_GLOBAL", "ID_ZONA", "NOMBRE_ZONA", "CLUSTER",
        "ESC_POTENCIAL_DEMANDA", "ESC_EFICIENCIA_FLUJO", "ESC_VIABILIDAD_RIESGO",
        "SCORE_MEDIO", "VARIABILIDAD_ENTRE_ESCENARIOS", "CONSISTENCIA_ESCENARIOS"
    ]].sort_values(by=["RANKING_GLOBAL", "VARIABILIDAD_ENTRE_ESCENARIOS", "NOMBRE_ZONA"])
)
pd.reset_option("display.max_rows")

print("✅ Consistencia interna y ranking global calculados correctamente.")
print("-" * 70)


In [ ]:
# PASO 6: CONSOLIDACIÓN FINAL

archivo_final = RUTA_RESULTADOS_ESCENARIOS / "consolidado_escenarios.csv"

columnas_maestras = [
    "ID_ZONA", "NOMBRE_ZONA", "CLUSTER",
    "ESC_POTENCIAL_DEMANDA", "RANK_DEMANDA",
    "ESC_EFICIENCIA_FLUJO", "RANK_FLUJO",
    "ESC_VIABILIDAD_RIESGO", "RANK_VIABILIDAD",
    "SCORE_MEDIO", "RANKING_GLOBAL",
    "VARIABILIDAD_ENTRE_ESCENARIOS", "CONSISTENCIA_ESCENARIOS"
]

df_esc[columnas_maestras].to_csv(archivo_final, index=False)
print(f"✅ Archivo maestro consolidado en: {archivo_final}")
